In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved
# as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session


/kaggle/input/datasets/seyedmahdivalizadeh/arman-dataset/test.tsv
/kaggle/input/datasets/seyedmahdivalizadeh/arman-dataset/train.tsv
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F13A20.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F23A06.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24N09.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24S05.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F21W01.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24S01.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F24A43.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-detection-database/female/F05S01.wav
/kaggle/input/datasets/mansourehk/shemo-persian-speech-emotion-dete

# Fine-tune XLM-RoBERTa on our own dataset

Second fine-tuning stage: starts from `maryamalikhasi/text-emotion-xlm-roberta-large`
(already trained on ArmanEmo) and fine-tunes it further on our own 615-sample
labeled Persian dataset. Final accuracy on a held-out set of transcripts: 86.5%.

In [2]:
# Model label set (single source of truth, reused in every cell below)
id2label = {
    0: "ANGRY",
    1: "FEAR",
    2: "HAPPY",
    3: "HATE",
    4: "OTHER",
    5: "SAD",
    6: "SURPRISE",
}
label2id = {v: k for k, v in id2label.items()}

print(label2id)


{'ANGRY': 0, 'FEAR': 1, 'HAPPY': 2, 'HATE': 3, 'OTHER': 4, 'SAD': 5, 'SURPRISE': 6}


In [3]:
df = pd.read_csv("/kaggle/input/datasets/ftmjfr/emotion-transcripts/emotion_dataset.csv")
df.head()


,text,label
0,اعصابم از صدای بوق ماشینا خورد شد,ANGER
1,یادمه شبی که تنها تو بیمارستان بودم و کسی کنار...,SADNESS
2,امروز برای جلسه دانشگاه یه گزارش کوتاه آماده کردم,NEUTRAL
3,امروز اولین باری بود که تونستم کیک بپزم و خیلی...,HAPPINESS
4,همون چند دقیقه برام مثل چند ساعت گذشت از ترس,FEAR


In [4]:
# Normalize label formatting
df["label"] = df["label"].str.upper().str.strip()

# Labels present in the dataset
dataset_labels = set(df["label"].unique())

# Labels expected by the model
model_labels = set(label2id.keys())

print("Dataset labels:")
print(dataset_labels)

print("\nModel labels:")
print(model_labels)

# Check for mismatches
extra_labels = dataset_labels - model_labels
missing_labels = model_labels - dataset_labels

if len(extra_labels) > 0:
    print("\nUnrecognized labels in the dataset:")
    print(extra_labels)

if len(missing_labels) > 0:
    print("\nModel labels missing from the dataset:")
    print(missing_labels)

if len(extra_labels) == 0:
    print("\nAll labels are compatible with the model.")


Dataset labels:
{'SADNESS', 'NEUTRAL', 'ANGER', 'FEAR', 'HAPPINESS'}

Model labels:
{'HAPPY', 'HATE', 'ANGRY', 'SURPRISE', 'OTHER', 'SAD', 'FEAR'}

Unrecognized labels in the dataset:
{'HAPPINESS', 'SADNESS', 'NEUTRAL', 'ANGER'}

Model labels missing from the dataset:
{'HAPPY', 'HATE', 'ANGRY', 'SURPRISE', 'OTHER', 'SAD'}


In [5]:
# Map dataset labels onto the model's label set
label_mapping = {
    "ANGER": "ANGRY",
    "HAPPINESS": "HAPPY",
    "SADNESS": "SAD",
    "FEAR": "FEAR",
    "NEUTRAL": "OTHER",
}

df["label"] = df["label"].map(label_mapping)

# Check that every label was successfully mapped
if df["label"].isna().sum() > 0:
    print("Some labels have no mapping:")
    print(df[df["label"].isna()]["label"])
else:
    print("All labels were mapped successfully.")

print(df["label"].value_counts())


All labels were mapped successfully.
label
FEAR     187
ANGRY    134
OTHER    100
HAPPY     99
SAD       95
Name: count, dtype: int64


In [6]:
from datasets import Dataset

# Convert label strings to numeric ids
df["label_id"] = df["label"].map(label2id)

print(df[["label", "label_id"]].head())
assert df["label_id"].isna().sum() == 0

# Build a Hugging Face Dataset
dataset = Dataset.from_pandas(df[["text", "label_id"]])

# Rename for the Trainer API
dataset = dataset.rename_column("label_id", "labels")

print(dataset)


   label  label_id
0  ANGRY         0
1    SAD         5
2  OTHER         4
3  HAPPY         2
4   FEAR         1
Dataset({
    features: ['text', 'labels'],
    num_rows: 615
})


In [7]:
from transformers import AutoTokenizer

model_name = "maryamalikhasi/text-emotion-xlm-roberta-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )


dataset = dataset.map(tokenize, batched=True)
dataset = dataset.remove_columns(["text"])
dataset.set_format("torch")


Map:   0%|          | 0/615 [00:00<?, ? examples/s]

In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

print(model.config.id2label)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

{0: 'ANGRY', 1: 'FEAR', 2: 'HAPPY', 3: 'HATE', 4: 'OTHER', 5: 'SAD', 6: 'SURPRISE'}


## Train

In [10]:
from datasets import ClassLabel

class_names = [id2label[i] for i in range(len(id2label))]  # ["ANGRY", "FEAR", "HAPPY", "HATE", "OTHER", "SAD", "SURPRISE"]
dataset = dataset.cast_column("labels", ClassLabel(names=class_names))

dataset_split = dataset.train_test_split(
    test_size=0.1,
    seed=42,
    stratify_by_column="labels",
)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

Casting the dataset:   0%|          | 0/615 [00:00<?, ? examples/s]

In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./emotion_model",

    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,

    eval_strategy="epoch",
    save_strategy="epoch", 

    load_best_model_at_end=True,
    save_total_limit=1,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_steps=50,
    report_to="none",
)


In [13]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)


In [14]:
trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,0.213751
2,0.736561,0.039982
3,0.065194,0.016573


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=105, training_loss=0.38197770986173835, metrics={'train_runtime': 154.7114, 'train_samples_per_second': 10.723, 'train_steps_per_second': 0.679, 'total_flos': 386525062764288.0, 'train_loss': 0.38197770986173835, 'epoch': 3.0})

In [15]:
import os
import json

# Save path for the fine-tuned model
save_path = "./emotion_xlm_roberta_final"
os.makedirs(save_path, exist_ok=True)

# Save model weights + config
model.save_pretrained(save_path)

# Save tokenizer
tokenizer.save_pretrained(save_path)

# Save label mapping (reusing the id2label/label2id defined earlier)
with open(f"{save_path}/label_mapping.json", "w", encoding="utf-8") as f:
    json.dump(
        {"id2label": id2label, "label2id": label2id},
        f,
        ensure_ascii=False,
        indent=4,
    )

print("Model saved successfully.")
print(f"Location: {save_path}")

print("\nSaved files:")
for file in os.listdir(save_path):
    print("-", file)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully.
Location: ./emotion_xlm_roberta_final

Saved files:
- label_mapping.json
- tokenizer_config.json
- tokenizer.json
- model.safetensors
- config.json


In [16]:
import shutil
shutil.rmtree("./emotion_model", ignore_errors=True)
print("Checkpoint directory removed.")

Checkpoint directory removed.


## Evaluate on held-out transcripts

In [17]:
import torch
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

# ==========================
# Map the test dataset's short filename labels onto the model's label set
# ==========================
test_label_map = {
    "ANG": "ANGRY",
    "FEA": "FEAR",
    "HAP": "HAPPY",
    "SAD": "SAD",
    "NEU": "OTHER",  # the model has no NEUTRAL class
}

test_path = "/kaggle/input/datasets/ftmjfr/audio-transcript/dataset/transcripts"

texts = []
true_labels = []

# ==========================
# Read the transcript files
# ==========================
for file in os.listdir(test_path):
    if not file.endswith(".txt"):
        continue

    label_short = file.split("_")[1].replace(".txt", "")
    if label_short not in test_label_map:
        continue

    with open(os.path.join(test_path, file), "r", encoding="utf-8") as f:
        text = f.read().strip()

    texts.append(text)
    true_labels.append(test_label_map[label_short])

print("Number of test samples:", len(texts))
print("Labels:", set(true_labels))

# ==========================
# Inference (batched, for speed)
# ==========================
model.eval()
pred_labels = []
BATCH_SIZE = 16

for start in tqdm(range(0, len(texts), BATCH_SIZE)):
    batch_texts = texts[start:start + BATCH_SIZE]

    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=256,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    batch_predictions = torch.argmax(outputs.logits, dim=1).tolist()
    pred_labels.extend(id2label[p] for p in batch_predictions)

# ==========================
# Results
# ==========================
print("\nAccuracy:")
print(accuracy_score(true_labels, pred_labels))

print("\nClassification Report:")
print(classification_report(true_labels, pred_labels, zero_division=0))

print("\nSample predictions:")
for i in range(min(10, len(texts))):
    print("--------------------")
    print("TEXT:", texts[i][:100])
    print("TRUE:", true_labels[i])
    print("PRED:", pred_labels[i])


Number of test samples: 37
Labels: {'HAPPY', 'ANGRY', 'OTHER', 'SAD', 'FEAR'}


100%|██████████| 3/3 [00:00<00:00,  9.39it/s]


Accuracy:
0.8648648648648649

Classification Report:
              precision    recall  f1-score   support

       ANGRY       0.69      1.00      0.82         9
        FEAR       1.00      0.33      0.50         3
       HAPPY       0.90      1.00      0.95         9
       OTHER       1.00      1.00      1.00         7
         SAD       1.00      0.67      0.80         9

    accuracy                           0.86        37
   macro avg       0.92      0.80      0.81        37
weighted avg       0.90      0.86      0.85        37


Sample predictions:
--------------------
TEXT: برای ناهار ماکارونی پختم
TRUE: OTHER
PRED: OTHER
--------------------
TEXT: هوا امروز ابریه فکر میکنم بارون بیاد
TRUE: OTHER
PRED: OTHER
--------------------
TEXT: وای باورم نمیشه یعنی واقعا این همه هدیه برای منه
TRUE: HAPPY
PRED: HAPPY
--------------------
TEXT: ببین باورم نمیشه بالاخره بعد از کلی تلاش به نتیجه دلخواهم رسیدم
TRUE: HAPPY
PRED: HAPPY
--------------------
TEXT: امروز با استادم بحثم شد و خیلی